# Amazon ML Challenge: France-adapted reranker "fr" (Kaggle, GPU T4 ×2), resumable

Continues **reranker v3** (e5-base) on a French-heavy set: 2.05M French pseudo-labelled pairs (confident positives +
their hardest candidates as negatives, and distractor records) + 600k labelled US/India pairs (no forgetting).
Then scores the validation pairs (for our stage 2) and the **France test pairs only**.

**Inputs to attach (Add Input):**
1. dataset from local `kaggle/ce_fr` (`ce_train_fr.parquet`)
2. dataset `ce_handoff3` (already uploaded: `ce_val_v3`, `ce_test_v3`)
3. **the output of the Amazon_Crazy_3 notebook** (reranker v3: `ce_model/` + `ce_val_full_scores_v3.parquet`)

**Settings:** GPU T4 x2, Internet on → Save Version → Save & Run All (Commit). About 2.5 h.
**Outputs to download:** `ce_val_full_scores_fr.parquet`, `ce_test_full_scores_fr.parquet` → project root.
Resume after a crash: add this notebook's previous version (its Output) as an input and commit again.

In [ ]:
import glob
import json
import os
import shutil
import subprocess
import time

import pyarrow.compute as pc
import pyarrow.parquet as pq

W = "/kaggle/working"
CFG = {"CE_MAXLEN": "96", "CE_INF_BS": "384", "CE_WORKERS": "1", "CE_EPOCHS": "1", "CE_SAVE_EVERY": "2000",
       "CE_BS": "64", "CE_TAG": "fr", "CE_TRAIN_FILE": "ce_train_fr.parquet", "CE_MAX_TRAIN": "0",
       "CE_TRAIN_MINUTES": "100", "CE_LR": "1e-5"}
OUTNAME = {"ce_val_v3": "ce_val_full_scores_fr.parquet", "ce_test_fr": "ce_test_full_scores_fr.parquet"}

# ---- data folder: training file, validation pairs, France-only test pairs
tr = glob.glob("/kaggle/input/**/ce_train_fr.parquet", recursive=True)
va = glob.glob("/kaggle/input/**/ce_val_v3.parquet", recursive=True)
te = glob.glob("/kaggle/input/**/ce_test_v3.parquet", recursive=True)
assert tr and va and te, "attach the ce_fr and ce_handoff3 datasets"
D = f"{W}/data"
os.makedirs(D, exist_ok=True)
for src_, name in ((tr[0], "ce_train_fr.parquet"), (va[0], "ce_val_v3.parquet")):
    if not os.path.exists(f"{D}/{name}"):
        os.symlink(src_, f"{D}/{name}")
if not os.path.exists(f"{D}/ce_test_fr.parquet"):
    t = pq.read_table(te[0])
    pq.write_table(t.filter(pc.equal(t.column("country"), "France")), f"{D}/ce_test_fr.parquet")
    print("France test pairs:", pq.ParquetFile(f"{D}/ce_test_fr.parquet").metadata.num_rows)
CFG["CE_DATA"], CFG["CE_OUT"] = D, W
CFG["CE_MODEL_DIR"], CFG["CE_CKPT_DIR"] = f"{W}/ce_model", f"{W}/ckpt"

# ---- starting model: reranker v3 (the ce_model next to ce_val_full_scores_v3.parquet in the Crazy_3 output)
v3 = [os.path.dirname(p) for p in glob.glob("/kaggle/input/**/ce_model/config.json", recursive=True)
      if glob.glob(os.path.join(os.path.dirname(os.path.dirname(p)), "ce_val_full_scores_v3.parquet"))]
assert v3, "attach the output of the Amazon_Crazy_3 notebook (reranker v3)"
CFG["CE_MODEL"] = v3[0]

# ---- resume from THIS notebook's previous version (never from the v3 model itself)
def _prev(pattern):
    return [p for p in glob.glob(f"/kaggle/input/**/{pattern}", recursive=True)]

for f in OUTNAME.values():
    for p in _prev(f):
        if not os.path.exists(f"{W}/{f}"):
            shutil.copy(p, f"{W}/{f}"); print("resumed finished scores", p)
for p in _prev("scores_*.partial.npz"):
    if not os.path.exists(f"{W}/{os.path.basename(p)}"):
        shutil.copy(p, W); print("resumed partial scores", p)
for p in _prev("ce_model/TAG"):
    if open(p).read().strip() == "fr" and not os.path.exists(f"{W}/ce_model/STATUS"):
        shutil.copytree(os.path.dirname(p), f"{W}/ce_model", dirs_exist_ok=True); print("resumed model", p)
for p in _prev("ckpt/state.pt"):
    if not os.path.exists(f"{W}/ckpt/state.pt") and not os.path.exists(f"{W}/ce_model/STATUS"):
        os.makedirs(f"{W}/ckpt", exist_ok=True); shutil.copy(p, f"{W}/ckpt/state.pt"); print("resumed checkpoint", p)

os.environ.update(CFG)
os.environ.update({"NCCL_P2P_DISABLE": "1", "NCCL_IB_DISABLE": "1", "TOKENIZERS_PARALLELISM": "false",
                   "PYTORCH_ALLOC_CONF": "expandable_segments:True", "PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True"})
print(CFG)
print(subprocess.run(["nvidia-smi", "--query-gpu=index,name,memory.total", "--format=csv"], capture_output=True, text=True).stdout)
import torch
NGPU = torch.cuda.device_count()
print("GPUs visible:", NGPU)

In [ ]:
%%writefile /kaggle/working/ce.py
"""Cross-encoder: fine-tune (DDP) and score pairs (one shard per GPU), with resumable checkpoints.
usage: torchrun --nproc_per_node=N ce.py train | infer <name>"""
import os
import sys
import time

import numpy as np
import pyarrow.compute as pc
import pyarrow.parquet as pq
import torch
import torch.distributed as dist
from torch.utils.data import DataLoader, Dataset
from transformers import AutoModelForSequenceClassification, AutoTokenizer, get_linear_schedule_with_warmup

E = os.environ
MODEL, DATA, OUT = E["CE_MODEL"], E["CE_DATA"], E["CE_OUT"]
MODEL_DIR = E.get("CE_MODEL_DIR", os.path.join(OUT, "ce_model"))
CKPT_DIR = E.get("CE_CKPT_DIR", os.path.join(OUT, "ckpt"))
TRAIN_FILE = E.get("CE_TRAIN_FILE", "ce_train.parquet")
MAXLEN, BS, INF_BS = int(E.get("CE_MAXLEN", 96)), int(E.get("CE_BS", 128)), int(E.get("CE_INF_BS", 512))
EPOCHS, LR, MAX_TRAIN = float(E.get("CE_EPOCHS", 1)), float(E.get("CE_LR", 6e-5)), int(E.get("CE_MAX_TRAIN", 0))
WORKERS, SAVE_EVERY = int(E.get("CE_WORKERS", 1)), int(E.get("CE_SAVE_EVERY", 1500))
DEADLINE = float(E.get("CE_DEADLINE", 0))               # absolute time.time() at which training stops (0 = none)
INF_SAVE = 1000                                          # scoring: save partial progress every INF_SAVE batches


def setup():
    world = int(E.get("WORLD_SIZE", 1))
    if world > 1:
        dist.init_process_group("nccl" if torch.cuda.is_available() and os.name != "nt" else "gloo")
    rank, local = int(E.get("RANK", 0)), int(E.get("LOCAL_RANK", 0))
    dev = torch.device(f"cuda:{local}" if torch.cuda.is_available() else "cpu")
    if dev.type == "cuda":
        torch.cuda.set_device(dev)
    return rank, world, dev


def log(rank, *a):
    if rank == 0:
        print(time.strftime("%H:%M:%S"), *a, flush=True)


class Batches(Dataset):
    """Item i = one pre-planned batch of row indices -> tokenised tensors (runs in a worker process)."""
    def __init__(self, tbl, batches, tok, y=None):
        self.tbl, self.batches, self.tok, self.y = tbl, batches, tok, y

    def __len__(self):
        return len(self.batches)

    def __getitem__(self, i):
        idx = np.sort(self.batches[i])
        sub = self.tbl.take(idx)
        x = dict(self.tok(sub.column("r_text").to_pylist(), sub.column("s_text").to_pylist(),
                          truncation="longest_first", max_length=MAXLEN, padding=True, return_tensors="pt"))
        x["_idx"] = torch.from_numpy(idx)
        if self.y is not None:
            x["_y"] = torch.from_numpy(self.y[idx])
        return x


def loader(ds):
    return DataLoader(ds, batch_size=None, shuffle=False, num_workers=WORKERS, pin_memory=True,
                      prefetch_factor=8 if WORKERS else None)


def lengths(tbl):
    return (pc.utf8_length(tbl.column("r_text")).to_numpy(zero_copy_only=False)
            + pc.utf8_length(tbl.column("s_text")).to_numpy(zero_copy_only=False))


def save_model(model, tok, world, note):
    m = model.module if world > 1 else model
    m.save_pretrained(MODEL_DIR)
    tok.save_pretrained(MODEL_DIR)
    open(os.path.join(MODEL_DIR, "STATUS"), "w").write(note)
    open(os.path.join(MODEL_DIR, "TAG"), "w").write(E.get("CE_TAG", ""))


def save_ckpt(model, opt, sched, scaler, pairs_done, world):
    """Full training state, written atomically (a crash while saving keeps the previous checkpoint)."""
    m = model.module if world > 1 else model
    os.makedirs(CKPT_DIR, exist_ok=True)
    tmp = os.path.join(CKPT_DIR, "state.pt.tmp")
    torch.save({"model": m.state_dict(), "opt": opt.state_dict(), "sched": sched.state_dict(),
                "scaler": scaler.state_dict(), "pairs_done": pairs_done, "bs": BS, "world": world}, tmp)
    os.replace(tmp, os.path.join(CKPT_DIR, "state.pt"))


def train(rank, world, dev):
    tbl = pq.read_table(os.path.join(DATA, TRAIN_FILE), columns=["r_text", "s_text", "label"])
    if MAX_TRAIN:
        tbl = tbl.slice(0, MAX_TRAIN)
    n = tbl.num_rows
    y = tbl.column("label").to_numpy().astype(np.float32)
    ln = lengths(tbl)
    tok = AutoTokenizer.from_pretrained(MODEL)
    model = AutoModelForSequenceClassification.from_pretrained(MODEL, num_labels=1)
    ck_path = os.path.join(CKPT_DIR, "state.pt")
    ck = torch.load(ck_path, map_location="cpu", weights_only=False) if os.path.exists(ck_path) else None
    if ck:
        model.load_state_dict(ck["model"])
    model = model.to(dev)
    if world > 1:
        model = torch.nn.parallel.DistributedDataParallel(model, device_ids=[dev.index] if dev.type == "cuda" else None)
    per_ep = n // (BS * world)
    total = int(per_ep * EPOCHS)
    # batch plan: shuffle, bucket by length inside blocks of 64 batches, shuffle the batch order. It depends only on
    # the epoch seed, so a resumed run (even with another batch size) continues through the same pair order.
    plan = []
    for ep in range(int(np.ceil(EPOCHS))):
        rng = np.random.default_rng(ep)
        perm = rng.permutation(n)[: per_ep * BS * world]
        blk, bs = BS * 64, []
        for i in range(0, len(perm), blk):
            b = perm[i:i + blk]
            b = b[np.argsort(ln[b], kind="stable")]
            bs += [b[j:j + BS] for j in range(0, len(b), BS)]
        plan += [bs[k] for k in rng.permutation(len(bs))]
    plan = plan[: total * world]
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
    scaler = torch.amp.GradScaler("cuda", enabled=dev.type == "cuda")
    start = 0
    if ck:
        opt.load_state_dict(ck["opt"])
        scaler.load_state_dict(ck["scaler"])
        start = min(ck["pairs_done"] // (BS * world), total)
    sched = get_linear_schedule_with_warmup(opt, int(0.06 * total), total)
    if ck and ck["bs"] == BS and ck["world"] == world:
        sched.load_state_dict(ck["sched"])
    else:
        for _ in range(start):            # new batch size: put the linear schedule at the same point
            sched.step()
    del ck
    plan = plan[start * world:][rank::world]
    log(rank, f"model {MODEL}  train pairs {n:,}  world={world}  batch {BS}/GPU  steps {start}->{total}"
              + (f"  (resumed at {start * BS * world:,} pairs)" if start else ""))
    lossf = torch.nn.BCEWithLogitsLoss()
    model.train()
    t0, run, step = time.time(), 0.0, start
    for step, x in enumerate(loader(Batches(tbl, plan, tok, y)), start + 1):
        x.pop("_idx")
        yt = x.pop("_y").to(dev, non_blocking=True)
        x = {k: v.to(dev, non_blocking=True) for k, v in x.items()}
        with torch.autocast(dev.type, dtype=torch.float16, enabled=dev.type == "cuda"):
            logit = model(**x).logits.squeeze(-1)
        loss = lossf(logit.float(), yt)
        opt.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(opt)
        scaler.update()
        sched.step()
        if step % 50 == 0 or step == start + 1:
            run = loss.item() if step == start + 1 else 0.8 * run + 0.2 * loss.item()
            if DEADLINE:                  # time budget: every rank stops at the same step
                over = torch.tensor([float(time.time() > DEADLINE)], device=dev)
                if world > 1:
                    dist.all_reduce(over, op=dist.ReduceOp.MAX)
                if over.item() > 0:
                    log(rank, f"time budget reached at step {step}/{total}")
                    break
        if step % 200 == 0 or step == total:
            el = time.time() - t0
            done = step - start
            log(rank, f"step {step}/{total} loss {run:.4f}  {done * BS * world / el:,.0f} pairs/s  "
                      f"eta {(total - step) * el / done / 60:.1f} min")
        if step % SAVE_EVERY == 0 and step < total:
            if rank == 0:
                save_ckpt(model, opt, sched, scaler, step * BS * world, world)
                log(rank, f"checkpoint at step {step} ({step * BS * world:,} pairs)")
            if world > 1:
                dist.barrier()
    if rank == 0:
        save_model(model, tok, world, f"complete {step}/{total}")
        if os.path.exists(CKPT_DIR):
            import shutil
            shutil.rmtree(CKPT_DIR, ignore_errors=True)   # the finished model supersedes the checkpoint
        log(rank, f"saved {MODEL_DIR}  ({(time.time() - t0) / 60:.1f} min)")


@torch.no_grad()
def infer(rank, world, dev, name):
    n = pq.ParquetFile(os.path.join(DATA, f"{name}.parquet")).metadata.num_rows
    lo, hi = n * rank // world, n * (rank + 1) // world
    tbl = pq.read_table(os.path.join(DATA, f"{name}.parquet"), columns=["r_text", "s_text"]).slice(lo, hi - lo)
    tok = AutoTokenizer.from_pretrained(MODEL_DIR)
    model = AutoModelForSequenceClassification.from_pretrained(MODEL_DIR).to(dev).eval()
    if dev.type == "cuda":
        model = model.half()
    order = np.argsort(lengths(tbl), kind="stable")          # length-sorted batches: little padding
    plan = [order[i:i + INF_BS] for i in range(0, len(order), INF_BS)]
    part = os.path.join(OUT, f"scores_{name}_{rank:02d}.partial.npz")
    out, b0 = np.zeros(len(order), dtype=np.float32), 0
    if os.path.exists(part):
        z = np.load(part)
        if len(z["out"]) == len(order) and int(z["inf_bs"]) == INF_BS:
            out, b0 = z["out"], int(z["b"])
            log(rank, f"{name}: resuming at batch {b0}/{len(plan)}")
    t0 = time.time()
    for b, x in enumerate(loader(Batches(tbl, plan[b0:], tok)), b0):
        idx = x.pop("_idx").numpy()
        x = {k: v.to(dev, non_blocking=True) for k, v in x.items()}
        out[idx] = model(**x).logits.squeeze(-1).float().cpu().numpy()
        if (b + 1) % INF_SAVE == 0:
            np.savez(part + ".tmp.npz", out=out, b=b + 1, inf_bs=INF_BS)
            os.replace(part + ".tmp.npz", part)
            el = time.time() - t0
            log(rank, f"{name}: {(b + 1) * INF_BS:,}/{len(order):,} per GPU  {world * (b + 1 - b0) * INF_BS / el:,.0f} pairs/s total  "
                      f"eta {(len(plan) - b - 1) * el / (b + 1 - b0) / 60:.1f} min")
    np.save(os.path.join(OUT, f"scores_{name}_{rank:02d}.npy"), out)
    if os.path.exists(part):
        os.remove(part)
    log(rank, f"{name}: done in {(time.time() - t0) / 60:.1f} min")


if __name__ == "__main__":
    rank, world, dev = setup()
    if sys.argv[1] == "train":
        train(rank, world, dev)
    else:
        infer(rank, world, dev, sys.argv[2])
    if world > 1:
        dist.barrier()
        dist.destroy_process_group()

In [ ]:
def run(*args, check=True):
    cmd = ["torchrun", f"--nproc_per_node={max(NGPU, 1)}", f"{W}/ce.py", *args]
    print(" ".join(cmd), flush=True)
    t, tail = time.time(), []
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        tail = (tail + [line])[-300:]
        if "Warning" not in line and "warn(" not in line and "Materializing" not in line:
            print(line, end="", flush=True)
    rc = p.wait()
    print(f"--- {args} exit {rc} after {(time.time() - t) / 60:.1f} min", flush=True)
    assert rc == 0 or not check, f"{args} failed"
    return rc, "".join(tail)


# training with automatic out-of-memory recovery: halve the batch, resume from the last checkpoint
status = f"{W}/ce_model/STATUS"
os.environ["CE_DEADLINE"] = str(time.time() + float(CFG["CE_TRAIN_MINUTES"]) * 60)
while not os.path.exists(status):
    rc, tail = run("train", check=False)
    if rc == 0:
        break
    if "OutOfMemoryError" in tail and int(os.environ["CE_BS"]) > 4:
        os.environ["CE_BS"] = str(int(os.environ["CE_BS"]) // 2)
        print(f"*** out of memory -> retrying from the last checkpoint with CE_BS={os.environ['CE_BS']}", flush=True)
        continue
    raise RuntimeError("training failed (not an out-of-memory error): see the log above")
print("model", os.environ["CE_MODEL_DIR"], "->", open(status).read())

In [ ]:
import numpy as np
import pandas as pd


def merge(name, cols):
    df = pd.read_parquet(os.path.join(CFG["CE_DATA"], f"{name}.parquet"), columns=cols)
    parts = sorted(glob.glob(f"{W}/scores_{name}_[0-9][0-9].npy"))
    df["ce_logit"] = np.concatenate([np.load(p) for p in parts])
    df.to_parquet(f"{W}/{OUTNAME[name]}", index=False)
    for p in parts:
        os.remove(p)


for name, cols in (("ce_val_v3", ["rid", "s1", "country", "label", "xgb_p"]), ("ce_test_fr", ["rid", "s1", "country"])):
    if not os.path.exists(f"{W}/{OUTNAME[name]}"):
        for attempt in range(3):
            rc, tail = run("infer", name, check=False)
            if rc == 0:
                break
            if "OutOfMemoryError" in tail:
                os.environ["CE_INF_BS"] = str(int(os.environ["CE_INF_BS"]) // 2)
                for p in glob.glob(f"{W}/scores_{name}_*.partial.npz"):
                    os.remove(p)
                print(f"*** out of memory while scoring -> CE_INF_BS={os.environ['CE_INF_BS']}", flush=True)
        assert rc == 0, f"scoring {name} failed"
        merge(name, cols)
print(sorted(f for f in os.listdir(W) if f.endswith(".parquet")))

## Quick check on validation

In [ ]:
from sklearn.metrics import roc_auc_score

val = pd.read_parquet(f"{W}/ce_val_full_scores_fr.parquet")
print(f"[fr] AUC  xgb {roc_auc_score(val.label, val.xgb_p):.5f}   ce {roc_auc_score(val.label, val.ce_logit):.5f}")